In [92]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time

In [93]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [94]:
characters_dict = {'name': [], 'stars': [], 'pulled_count': [], 'constellation_0_pull': [], 'constellation_1_pull': [], 'constellation_2_pull': [], 'constellation_3_pull': [], 'constellation_4_pull': [], 'constellation_5_pull': [], 'constellation_6_pull': [], 'constellation_7_pull': [], 'special_skill': [], 'lvl_1_HP': [], 'lvl_1_ATK': [], 'lvl_1_DEF': [], 'asc_0_special_skill': [], 'lvl_20_Min_HP': [], 'lvl_20_Min_ATK': [], 'lvl_20_Min_DEF': [], 'lvl_20_Max_HP': [], 'lvl_20_Max_ATK': [], 'lvl_20_Max_DEF': [], 'asc_1_special_skill': [], 'lvl_40_Min_HP': [], 'lvl_40_Min_ATK': [], 'lvl_40_Min_DEF': [], 'lvl_40_Max_HP': [], 'lvl_40_Max_ATK': [], 'lvl_40_Max_DEF': [], 'asc_2_special_skill': [], 'lvl_50_Min_HP': [], 'lvl_50_Min_ATK': [], 'lvl_50_Min_DEF': [], 'lvl_50_Max_HP': [], 'lvl_50_Max_ATK': [], 'lvl_50_Max_DEF': [], 'asc_3_special_skill': [], 'lvl_60_Min_HP': [], 'lvl_60_Min_ATK': [], 'lvl_60_Min_DEF': [], 'lvl_60_Max_HP': [], 'lvl_60_Max_ATK': [], 'lvl_60_Max_DEF': [], 'asc_4_special_skill': [], 'lvl_70_Min_HP': [], 'lvl_70_Min_ATK': [], 'lvl_70_Min_DEF': [], 'lvl_70_Max_HP': [], 'lvl_70_Max_ATK': [], 'lvl_70_Max_DEF': [], 'asc_5_special_skill': [], 'lvl_80_Min_HP': [], 'lvl_80_Min_ATK': [], 'lvl_80_Min_DEF': [], 'lvl_80_Max_HP': [], 'lvl_80_Max_ATK': [], 'lvl_80_Max_DEF': [], 'asc_6_special_skill': [], 'lvl_90_HP': [], 'lvl_90_ATK': [], 'lvl_90_DEF': [], 'lvl_95_HP': [], 'lvl_95_ATK': [], 'lvl_95_DEF': [], 'lvl_100_HP': [], 'lvl_100_ATK': [], 'lvl_100_DEF': []}

In [95]:
with requests.Session() as session:
    count = 300093
    while count > 300092:
        wish_tally_response = session.get(f'https://api.paimon.moe/wish?banner={count}')
        time.sleep(0.5)
        
        if wish_tally_response.status_code != 200:
            print(f"Error {wish_tally_response.status_code}")
            continue
            
        try:
            wish_tally_response_data = wish_tally_response.json()
            for item in wish_tally_response_data.get('list', []):
                if item['type'] != 'character':
                    continue
                    
                if item['name'] in characters_dict['name']:
                    idx = characters_dict['name'].index(item['name'])
                    characters_dict['pulled_count'][idx] += item['count']
                    continue
                    
                characters_dict['name'].append(item['name'])
                
                characters_dict['pulled_count'].append(item['count'])

                if 'constellation' in wish_tally_response_data and item['name'] in wish_tally_response_data['constellation']:
                    constellation_data = wish_tally_response_data['constellation'][item['name']]
                    
                    length = len(constellation_data)
                    constellation_count = 0
                    while constellation_count < 8:
                        if constellation_count < length:
                            characters_dict['constellation_'+str(constellation_count)+'_pull'].append(constellation_data[constellation_count])
                        else:
                            characters_dict['constellation_'+str(constellation_count)+'_pull'].append(0)
                        constellation_count += 1
                else:
                    characters_dict['constellation_0_pull'].append(0)
                    characters_dict['constellation_1_pull'].append(0)
                    characters_dict['constellation_2_pull'].append(0)
                    characters_dict['constellation_3_pull'].append(0)
                    characters_dict['constellation_4_pull'].append(0)
                    characters_dict['constellation_5_pull'].append(0)
                    characters_dict['constellation_6_pull'].append(0)
                    characters_dict['constellation_7_pull'].append(0)
                
                character_info_response = session.get(f"https://paimon.moe/characters/{item['name']}")
                
                if character_info_response.status_code != 200:
                    print(f"Error {character_info_response.status_code}")
                    continue
                    
                try:
                    character_info_response_data = character_info_response.text
                    soup = BeautifulSoup(character_info_response_data, 'html.parser')
                    
                    stars_container = soup.find('div', class_='text-rare-from px-4 md:px-8 text-2xl flex items-center z-0 -mt-2 md:-mt-4 svelte-ti79zj')
                    if stars_container:
                        stars = stars_container.find_all('svg', class_='svelte-1mzwbk9')
                    else:
                        stars_container = soup.find('div', class_='text-legendary-from px-4 md:px-8 text-2xl flex items-center z-0 -mt-2 md:-mt-4 svelte-ti79zj')
                        stars = stars_container.find_all('svg', class_='svelte-1mzwbk9')
                    characters_dict['stars'].append(len(stars)-1)
                        
                    special_skill = soup.find_all('td', class_='text-center whitespace-nowrap border-gray-700 border-r font-semibold px-2 svelte-ti79zj')[5].text.strip()
                    characters_dict['special_skill'].append(special_skill)
                    stats = soup.find_all('td', class_='text-center border-t border-gray-700 border-r px-2 svelte-ti79zj')
                    
                    characters_dict['lvl_1_HP'].append(stats[1].text.strip())
                    characters_dict['lvl_1_ATK'].append(stats[2].text.strip())
                    characters_dict['lvl_1_DEF'].append(stats[3].text.strip())
                    characters_dict['asc_0_special_skill'].append(stats[4].text.strip())
                    
                    characters_dict['lvl_20_Min_HP'].append(stats[5].text.strip())
                    characters_dict['lvl_20_Min_ATK'].append(stats[6].text.strip())
                    characters_dict['lvl_20_Min_DEF'].append(stats[7].text.strip())
                    characters_dict['lvl_20_Max_HP'].append(stats[9].text.strip())
                    characters_dict['lvl_20_Max_ATK'].append(stats[10].text.strip())
                    characters_dict['lvl_20_Max_DEF'].append(stats[11].text.strip())
                    characters_dict['asc_1_special_skill'].append(stats[12].text.strip())
                    
                    characters_dict['lvl_40_Min_HP'].append(stats[13].text.strip())
                    characters_dict['lvl_40_Min_ATK'].append(stats[14].text.strip())
                    characters_dict['lvl_40_Min_DEF'].append(stats[15].text.strip())
                    characters_dict['lvl_40_Max_HP'].append(stats[17].text.strip())
                    characters_dict['lvl_40_Max_ATK'].append(stats[18].text.strip())
                    characters_dict['lvl_40_Max_DEF'].append(stats[19].text.strip())
                    characters_dict['asc_2_special_skill'].append(stats[20].text.strip())
                    
                    characters_dict['lvl_50_Min_HP'].append(stats[21].text.strip())
                    characters_dict['lvl_50_Min_ATK'].append(stats[22].text.strip())
                    characters_dict['lvl_50_Min_DEF'].append(stats[23].text.strip())
                    characters_dict['lvl_50_Max_HP'].append(stats[25].text.strip())
                    characters_dict['lvl_50_Max_ATK'].append(stats[26].text.strip())
                    characters_dict['lvl_50_Max_DEF'].append(stats[27].text.strip())
                    characters_dict['asc_3_special_skill'].append(stats[28].text.strip())
                    
                    characters_dict['lvl_60_Min_HP'].append(stats[29].text.strip())
                    characters_dict['lvl_60_Min_ATK'].append(stats[30].text.strip())
                    characters_dict['lvl_60_Min_DEF'].append(stats[31].text.strip())
                    characters_dict['lvl_60_Max_HP'].append(stats[33].text.strip())
                    characters_dict['lvl_60_Max_ATK'].append(stats[34].text.strip())
                    characters_dict['lvl_60_Max_DEF'].append(stats[35].text.strip())
                    characters_dict['asc_4_special_skill'].append(stats[36].text.strip())
                    
                    characters_dict['lvl_70_Min_HP'].append(stats[37].text.strip())
                    characters_dict['lvl_70_Min_ATK'].append(stats[38].text.strip())
                    characters_dict['lvl_70_Min_DEF'].append(stats[39].text.strip())
                    characters_dict['lvl_70_Max_HP'].append(stats[41].text.strip())
                    characters_dict['lvl_70_Max_ATK'].append(stats[42].text.strip())
                    characters_dict['lvl_70_Max_DEF'].append(stats[43].text.strip())
                    characters_dict['asc_5_special_skill'].append(stats[44].text.strip())
                    
                    characters_dict['lvl_80_Min_HP'].append(stats[45].text.strip())
                    characters_dict['lvl_80_Min_ATK'].append(stats[46].text.strip())
                    characters_dict['lvl_80_Min_DEF'].append(stats[47].text.strip())
                    characters_dict['lvl_80_Max_HP'].append(stats[49].text.strip())
                    characters_dict['lvl_80_Max_ATK'].append(stats[50].text.strip())
                    characters_dict['lvl_80_Max_DEF'].append(stats[51].text.strip())
                    characters_dict['asc_6_special_skill'].append(stats[52].text.strip())
                    
                    characters_dict['lvl_90_HP'].append(stats[53].text.strip())
                    characters_dict['lvl_90_ATK'].append(stats[54].text.strip())
                    characters_dict['lvl_90_DEF'].append(stats[55].text.strip())
                    characters_dict['lvl_95_HP'].append(stats[56].text.strip())
                    characters_dict['lvl_95_ATK'].append(stats[57].text.strip())
                    characters_dict['lvl_95_DEF'].append(stats[58].text.strip())
                    characters_dict['lvl_100_HP'].append(stats[59].text.strip())
                    characters_dict['lvl_100_ATK'].append(stats[60].text.strip())
                    characters_dict['lvl_100_DEF'].append(stats[61].text.strip())
                except Exception as e:
                    print(f"Error parsing data: {e}")
                    continue
        except Exception as e:
            print(f"{e}")
            continue
        print(f"Finished scouting page: {count}")
        count = count -1

Finished scouting page: 300093


In [96]:
df = pd.DataFrame(characters_dict)
df

,name,stars,pulled_count,constellation_0_pull,constellation_1_pull,constellation_2_pull,constellation_3_pull,constellation_4_pull,constellation_5_pull,constellation_6_pull,constellation_7_pull,special_skill,lvl_1_HP,lvl_1_ATK,lvl_1_DEF,asc_0_special_skill,lvl_20_Min_HP,lvl_20_Min_ATK,lvl_20_Min_DEF,lvl_20_Max_HP,lvl_20_Max_ATK,lvl_20_Max_DEF,asc_1_special_skill,lvl_40_Min_HP,lvl_40_Min_ATK,lvl_40_Min_DEF,lvl_40_Max_HP,lvl_40_Max_ATK,lvl_40_Max_DEF,asc_2_special_skill,lvl_50_Min_HP,lvl_50_Min_ATK,lvl_50_Min_DEF,lvl_50_Max_HP,lvl_50_Max_ATK,lvl_50_Max_DEF,asc_3_special_skill,lvl_60_Min_HP,lvl_60_Min_ATK,lvl_60_Min_DEF,lvl_60_Max_HP,lvl_60_Max_ATK,lvl_60_Max_DEF,asc_4_special_skill,lvl_70_Min_HP,lvl_70_Min_ATK,lvl_70_Min_DEF,lvl_70_Max_HP,lvl_70_Max_ATK,lvl_70_Max_DEF,asc_5_special_skill,lvl_80_Min_HP,lvl_80_Min_ATK,lvl_80_Min_DEF,lvl_80_Max_HP,lvl_80_Max_ATK,lvl_80_Max_DEF,asc_6_special_skill,lvl_90_HP,lvl_90_ATK,lvl_90_DEF,lvl_95_HP,lvl_95_ATK,lvl_95_DEF,lvl_100_HP,lvl_100_ATK,lvl_100_DEF
0,varesa,5,7832,4737,1188,756,316,190,228,399,18,CRIT Rate,989,28,61,5%,2564,72,158,3412,96,210,5%,5105,143,314,5708,160,351,9.8%,6567,184,404,7370,207,454,14.6%,8238,231,507,8840,248,544,14.6%,9716,273,598,10318,290,635,19.4%,11204,314,690,11806,331,727,24.2%,12699,356,782,13150,396,809,13602,437,837
1,chongyun,4,217,215,2,0,0,0,0,0,0,ATK%,921,19,54,0%,2366,48,140,3054,62,180,0%,4574,93,270,5063,103,299,6%,5824,118,344,6475,131,382,12%,7236,147,427,7725,157,456,12%,8485,172,501,8974,182,530,18%,9734,198,575,10223,208,603,24%,10984,223,648,11363,251,671,11743,280,693
2,tighnari,5,937,905,26,6,0,0,0,0,0,Dendro DMG Bonus,845,21,49,0%,2191,54,127,2915,72,169,0%,4362,108,253,4877,120,283,7.2%,5611,139,326,6297,155,366,14.4%,7038,174,409,7553,186,439,14.4%,8301,205,482,8816,218,512,21.6%,9573,236,556,10087,249,586,28.8%,10850,268,630,11235,298,653,11621,328,675
3,jean,5,922,888,34,0,0,0,0,0,0,Healing Bonus,1144,19,60,0%,2967,48,155,3948,64,206,0%,5908,96,309,6605,108,345,5.54%,7599,124,397,8528,139,446,11.08%,9533,155,499,10230,166,535,11.08%,11243,183,588,11940,194,624,16.62%,12965,211,678,13662,222,715,22.15%,14695,239,769,15217,266,796,15740,293,823
4,keqing,5,842,800,42,0,0,0,0,0,0,CRIT DMG,1020,25,62,50%,2646,65,161,3521,87,215,50%,5268,130,321,5889,145,359,59.6%,6776,167,413,7604,187,464,69.2%,8500,209,519,9121,225,556,69.2%,10025,247,612,10647,262,649,78.8%,11561,285,705,12182,300,743,88.4%,13103,323,799,13568,359,828,14034,396,856
5,gorou,4,185,183,2,0,0,0,0,0,0,Geo DMG Bonus,802,15,54,0%,2061,39,140,2661,51,180,0%,3985,76,270,4411,84,299,6%,5074,97,344,5642,108,382,12%,6305,120,427,6731,128,456,12%,7393,141,501,7818,149,530,18%,8481,162,575,8907,170,603,24%,9570,183,648,9901,206,671,10232,229,693
6,layla,4,222,208,14,0,0,0,0,0,0,HP%,930,18,55,0%,2389,47,141,3084,60,182,0%,4619,90,273,5113,100,302,6%,5881,115,347,6540,128,386,12%,7308,143,432,7801,152,461,12%,8569,167,506,9062,177,535,18%,9831,192,581,10324,202,610,24%,11092,217,655,11476,244,678,11860,272,701
7,iansan,4,37303,5999,6618,5760,4216,3345,2700,1673,6992,ATK%,894,22,54,0%,2296,55,137,2963,71,177,0%,4438,107,266,4913,118,294,6%,5651,136,338,6283,152,376,12%,7021,169,420,7495,181,449,12%,8233,199,493,8707,210,521,18%,9445,228,566,9919,239,594,24%,10657,257,638,11026,290,660,11395,323,682
8,gaming,4,37058,5923,6252,5703,4556,3345,2520,1911,6848,ATK%,957,25,59,0%,2460,65,151,3175,84,195,0%,4755,126,293,5264,139,324,6%,6054,160,373,6732,178,414,12%,7523,199,463,8031,212,494,12%,8821,233,543,9329,246,574,18%,10120,267,623,10628,281,654,24%,11419,302,703,11813,340,727,12208,379,752
9,noelle,4,225,219,6,0,0,0,0,0,0,DEF%,1012,16,67,0%,2600,41,172,3356,53,222,0%,5027,80,333,5564,88,368,7.5%,6400,101,423,7117,113,471,15%,7953,126,526,8490,134,562,15%,9325,148,617,9862,156,652,22.5%,10698,169,708,11235,178,743,30%,12071,191,799,12488,216,826,12906,240,854


In [97]:
df.to_csv("uncleaned_data.csv", encoding='utf-8')